In [1]:
#%%capture
#!wget "https://repo1.maven.org/maven2/org/apache/spark/spark-sql-kafka-0-10_2.12/3.5.0/spark-sql-kafka-0-10_2.12-3.5.0.jar"
#!wget "https://repo1.maven.org/maven2/org/apache/spark/spark-streaming-kafka-0-10_2.12/3.5.0/spark-streaming-kafka-0-10_2.12-3.5.0.jar"

In [2]:
import os
os.environ['PYSPARK_SUBMIT_ARGS'] = '--packages org.apache.spark:spark-streaming-kafka-0-10_2.12:3.5.0,org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.0 pyspark-shell'

In [3]:
!pip install mistralai==1.9.7

In [6]:
#Need to restart kernel once after finishing install mistral

In [4]:
import pandas as pd
import os
import pickle
import json
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *

spark = SparkSession.builder.appName("OCR").getOrCreate()
spark.sparkContext.setLogLevel("WARN")

In [6]:
# 1. Read Stream from Kafka
raw_stream = spark \
  .readStream \
  .format("kafka") \
  .option("kafka.bootstrap.servers", "192.168.1.124:8097") \
  .option("subscribe", "input") \
  .option("startingOffsets", "latest") \
  .load()

raw_stream.printSchema()

root
 |-- key: binary (nullable = true)
 |-- value: binary (nullable = true)
 |-- topic: string (nullable = true)
 |-- partition: integer (nullable = true)
 |-- offset: long (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- timestampType: integer (nullable = true)



In [7]:
# For debugging
if False:
    query = raw_stream \
      .selectExpr("CAST(key AS STRING)") \
      .writeStream \
      .outputMode("update") \
      .format("console") \
      .option("truncate", False) \
      .start()
    
    query.awaitTermination()

In [8]:
processed_data = raw_stream.select(
    col("key").cast(StringType()).alias("filename"),
    col("value").alias("png_data")
)

In [9]:
#import requests
from pyspark.sql import Row
from mistralai import DocumentURLChunk, ImageURLChunk, TextChunk
import json
import base64
import os
import time
# ---------------------

# Initialize Mistral client with API key
from mistralai import Mistral
api_key = ''
client = Mistral(api_key=api_key)

In [48]:
def call_mistral_ocr(row: Row):

    filename = row['filename']
    png_bytes = row['png_data']
    
    # Encode image as base64 for API
    encoded = base64.b64encode(png_bytes).decode()
    base64_data_url = f"data:image/jpeg;base64,{encoded}"
    #print(base64_data_url)
    
    # Process image with OCR
    image_response = client.ocr.process(
        document=ImageURLChunk(image_url=base64_data_url),
        model="mistral-ocr-latest"
    )

    return filename, image_response, base64_data_url
    # Convert response to JSON
    #response_dict = json.loads(image_response.model_dump_json())
    #ocr_result = json.dumps(response_dict, indent=4)
    #print(json_string)
    #ocr_result = json_string
    #return filename, ocr_result

In [49]:
def call_mistral_8b_latest(row: Row):

    filename = row['filename']
    png_bytes = row['png_data']
    
    # Encode image as base64 for API
    encoded = base64.b64encode(png_bytes).decode()
    base64_data_url = f"data:image/jpeg;base64,{encoded}"
    #print(base64_data_url)
    
    # Process image with OCR
    image_response = client.ocr.process(
        document=ImageURLChunk(image_url=base64_data_url),
        model="mistral-ocr-latest"
    )

    image_ocr_markdown = image_response.pages[0].markdown

    chat_response = client.chat.complete(
        #model="ministral-8b-latest",
        model = "pixtral-12b-latest",
        messages=[
            {
                "role": "user",
                "content": [
                    TextChunk(
                        text=(
                            f"This is image's OCR in markdown:\n\n{image_ocr_markdown}\n.\n"
                            "Convert this into a sensible structured json response. "
                            "The output should be strictly be json with no extra commentary"
                        )
                    ),
                ],
            }
        ],
        response_format={"type": "json_object"},
        temperature=0,
    )

    response_dict = json.loads(chat_response.choices[0].message.content)
    return filename, json.dumps(response_dict, indent=4)

In [ ]:
# Still got error.

In [50]:
from enum import Enum
from pathlib import Path
from pydantic import BaseModel
import base64

class StructuredOCR(BaseModel):
    file_name: str
    topics: list[str]
    languages: str
    ocr_contents: dict

def structured_ocr(row: Row) -> StructuredOCR:
    
    filename, image_ocr_markdown, base64_data_url = call_mistral_ocr(row)
    #print(filename)
    #print(image_ocr_markdown)
    #print(base64_data_url)

    # Parse the OCR result into a structured JSON response
    chat_response = client.chat.parse(
        model="pixtral-12b-latest",
        messages=[
            {
                "role": "user",
                "content": [
                    ImageURLChunk(image_url=base64_data_url),
                    TextChunk(text=(
                        f"This is the image's OCR in markdown:\n{image_ocr_markdown}\n.\n"
                        "Convert this into a structured JSON response "
                        "with the OCR contents in a sensible dictionnary."
                        )
                    )
                ]
            }
        ],
        response_format=StructuredOCR,
        temperature=0
    )

    print(chat_response)

    return filename, chat_response.choices[0].message.parsed

In [53]:
def process_batch(df, batch_id):
    """
    Function executed for every micro-batch of the Structured Stream.
    Collects the rows and processes them with the OCR function.
    """

    start_time_full = time.time()
    
    # ⚠️ WARNING: .collect() brings ALL data to the driver program. 
    # Use for testing/low-volume only. For high-volume, consider a custom Kafka Connect Sink.
    rows = df.collect()
    
    results = []
    c = 1
    for row in rows:
        print("c = ",c)
        #filename, ocr_text = call_mistral_ocr(row)
        filename, ocr_text = call_mistral_8b_latest(row)
        #filename, orc_text = structured_ocr(row)
        #filename, structured_response = structured_ocr(row) # Process image and extract data
        #response_dict = json.loads(structured_response.model_dump_json())
        #print(json.dumps(response_dict, indent=4))
        
        if filename is None:
            filename = f"image_{batch_id}"
        results.append((filename, ocr_text))        
        print(f"OCR Result for {filename}: {ocr_text}") # Log first 80 chars
        c = c+1

    schema = StructType([
        StructField("filename", StringType(), False),
        StructField("ocr_text", StringType(), False) 
    ])

    if results:
        results_df = spark.createDataFrame(results, schema)

        # 3. Write results to JSON files (The Solution)
        # Use the batch_id to create a unique subdirectory for the JSON files
        batch_output_path = os.path.join("/home/jovyan/json", f"batch_{batch_id}")

        results_df.write \
            .format("json") \
            .mode("overwrite") \
            .save(batch_output_path)
        
        print(f"Successfully wrote OCR results for Batch {batch_id} to: {batch_output_path}")
        
        end_time_full = time.time()
        
        total_duration = end_time_full - start_time_full
        
        print(f"   > Total Function Duration: {total_duration:.4f} seconds")
    

In [54]:
query = processed_data.writeStream \
    .foreachBatch(process_batch) \
    .start() 

query.awaitTermination()

c =  1
OCR Result for receipt1.png: {
    "parking_permit": {
        "instructions": {
            "display_instruction": "PLACE FACE UP ON DASH",
            "final_display_instruction": "DISPLAY FACE UP ON DASH"
        },
        "location": {
            "city": "Palo Alto"
        },
        "restrictions": {
            "validity": {
                "on_street_parking": false
            }
        },
        "expiration": {
            "date": "2024-08-19",
            "time": "23:59:59",
            "note": "Permit expires at midnight"
        },
        "transaction": {
            "purchase": {
                "date": "2024-08-19",
                "time": "13:34:00",
                "total_due": 15.0,
                "total_paid": 15.0
            },
            "ticket_details": {
                "ticket_number": "00005883",
                "serial_number": "520117260957"
            },
            "payment": {
                "type": "Credit Card (Swipe)",
                "

ERROR:root:KeyboardInterrupt while sending command.
Traceback (most recent call last):
  File "/usr/local/spark/python/lib/py4j-0.10.9.7-src.zip/py4j/java_gateway.py", line 1038, in send_command
    response = connection.send_command(command)
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/spark/python/lib/py4j-0.10.9.7-src.zip/py4j/clientserver.py", line 511, in send_command
    answer = smart_decode(self.stream.readline()[:-1])
                          ^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/conda/lib/python3.11/socket.py", line 706, in readinto
    return self._sock.recv_into(b)
           ^^^^^^^^^^^^^^^^^^^^^^^
KeyboardInterrupt

KeyboardInterrupt



In [27]:
for q in spark.streams.active:
    q.stop()

In [ ]:
#Mistral tutorial https://colab.research.google.com/github/mistralai/cookbook/blob/main/mistral/ocr/structured_ocr.ipynb#scrollTo=po7Cukllt8za